In [ ]:
import pyspark 
from pyspark.sql import SparkSession

spark = SparkSession.builder \
.appName("spark_application")\
.master("local[*]")\
.getOrCreate()

emp_data = [(1,'A',20000),(2,'B',23000),(3,'c',250000),(4,'D',130000)]
emp_df = spark.createDataFrame(emp_data, ['Id', 'name', 'Salary'])
emp_df.show()


### Introduction to Apache Spark

Apache Spark is a multi-language, open-source analytical engine designed for large-scale data processing. It handles various workloads, including data engineering, data science, and machine learning. While it can run on a single machine, its primary power lies in distributed data processing across a cluster (a group of computers), allowing it to handle massive datasets much faster than traditional methods.

### Limitations of Hadoop MapReduce

Before Spark, MapReduce was the primary tool for big data, but it faced several challenges:

* **Lack of Interactivity:** MapReduce is not suitable for interactive queries; there is a significant delay between submitting a query and receiving a result.
* **Inefficient Iterative Workloads:** For tasks like machine learning that require processing the same data repeatedly (iterations), MapReduce is very slow because it writes data back to the hard disk after every step.
* **Coding Complexity:** MapReduce code is typically written in Java and is very long and complex, making it difficult to develop and maintain.

### Performance Benchmarks

Spark is significantly faster than MapReduce. In the "GraySort" competition involving 100 terabytes of data, Spark completed the task in 23 minutes using 206 nodes, while MapReduce took 72 minutes and required 2,100 nodes. This demonstrates that Spark can be up to 100 times faster while using 10 times fewer resources.

### Spark vs. Hadoop: Not a Direct Replacement

A common misconception is that Spark replaces Hadoop entirely. In reality, Spark only replaces the **MapReduce** component of Hadoop. Spark can still utilize Hadoop's storage (HDFS) and resource manager (YARN). Spark is flexible and can run on various cluster managers, including YARN, Mesos, Kubernetes, and its own standalone scheduler.

### The Secret to Spark's Speed: In-Memory Processing

The primary reason Spark outperforms MapReduce is **in-memory processing**. MapReduce writes intermediate results to the hard disk, which has a slow transfer rate. In contrast, Spark keeps intermediate data in RAM whenever possible. Since RAM transfer speeds (around 10 GB/s) are vastly superior to hard disk speeds (around 100 MB/s), the overall processing time is drastically reduced.

### Data Storage Philosophy

Spark is a processing engine, not a storage system. It does not store data long-term like a database (e.g., MySQL or Hive). It is comparable to a water processing plant: water (data) flows through it to be cleaned (processed) but is stored in separate tanks (external storage like HDFS, S3, or NoSQL databases).

### The Unified Stack and Language Support

Spark offers a "Unified Stack," meaning a single installation provides tools for SQL, Machine Learning (MLlib), Graph processing (GraphX), and Real-time streaming. It supports five major programming languages:

1. **Scala** (The native language of Spark)
2. **Python** (Used via PySpark, very popular in data science)
3. **Java**
4. **R**
5. **SQL** (Available across all other language interfaces)

### **Core Concepts of Spark Architecture**

* **Cluster View:** This represents the physical setup where multiple computers are grouped to form a cluster. One computer acts as the **Master**, while the others are **Slaves**. The primary purpose of the cluster is to provide resources like RAM and processors for applications to run.
* **Application View:** Every Spark application has its own dedicated internal structure, consisting of a **Driver** (the master process) and **Executors** (the slave processes). These are Java processes running on the cluster, not physical machines themselves.
* **The Merged View:** In a real-world scenario, a single physical cluster can host multiple Spark applications simultaneously. Each application runs its own set of drivers and executors across the worker machines provided by the cluster.

### **Cluster Managers (The "Pluggable" Component)**

Spark applications are flexible and can be deployed using different cluster managers. The video likens this to a "ring light" that can be plugged into different "sockets" (cluster managers) without changing the core code:

* **Standalone Scheduler:** Running Spark on its own dedicated Spark cluster.
* **Hadoop YARN:** A popular traditional method for deploying Spark on Hadoop clusters.
* **Kubernetes:** A modern, cloud-native approach where drivers and executors run in separate containers.



**RDD -- Resilient Distrbuted Datasets**

RDDs' in spark are fundamental data elements that are immutable -- which can't be chagned once created. On performing a new operation results in creating a new RDD. They will be devided into partitions and allows parallel execution.
Operations performed on RDDs' are called as Transformations. They are 2 types 
1. Narrow Transformations 
2. wide Transformations 

***NARROW Transformations***
(No shuffle)

- map
- flatMap
- filter
- mapPartitions
- union

***WIDE Transformations***
(Shuffle happens)

- reduceByKey ⭐
- groupByKey ❌ (know it, avoid it)
- distinct
- join
- repartition
- sortBy / sortByKey

Tasks that are performed for fetching the result are called Actions.

***Tasks***
- collect ⚠️
- count
- take
- first
- reduce
- foreach
- foreachPartition
- takeSample

![alt text](image.png)

In [2]:
orders_rdd = spark.sparkContext.parallelize([
    (1, 101, "Electronics", 1200, "2025-01-01"),
    (2, 102, "Clothing", 700, "2025-01-01"),
    (3, 103, "Electronics", 300, "2025-01-02"),
    (4, 101, "Home", 1500, "2025-01-02"),
    (5, 104, "Clothing", 400, "2025-01-03"),
    (6, 105, "Electronics", 2200, "2025-01-03"),
    (7, 102, "Home", 800, "2025-01-04"),
    (8, 106, "Books", 250, "2025-01-04"),
    (9, 107, "Books", 600, "2025-01-05"),
    (10, 108, "Clothing", 1100, "2025-01-05"),
    (11, 101, "Electronics", 900, "2025-01-06"),
    (12, 109, "Home", 300, "2025-01-06"),
    (13, 110, "Clothing", 950, "2025-01-07"),
    (14, 111, "Electronics", 500, "2025-01-07"),
    (15, 112, "Books", 1300, "2025-01-08")
])

In [3]:
orders_rdd.collect()

[(1, 101, 'Electronics', 1200, '2025-01-01'),
 (2, 102, 'Clothing', 700, '2025-01-01'),
 (3, 103, 'Electronics', 300, '2025-01-02'),
 (4, 101, 'Home', 1500, '2025-01-02'),
 (5, 104, 'Clothing', 400, '2025-01-03'),
 (6, 105, 'Electronics', 2200, '2025-01-03'),
 (7, 102, 'Home', 800, '2025-01-04'),
 (8, 106, 'Books', 250, '2025-01-04'),
 (9, 107, 'Books', 600, '2025-01-05'),
 (10, 108, 'Clothing', 1100, '2025-01-05'),
 (11, 101, 'Electronics', 900, '2025-01-06'),
 (12, 109, 'Home', 300, '2025-01-06'),
 (13, 110, 'Clothing', 950, '2025-01-07'),
 (14, 111, 'Electronics', 500, '2025-01-07'),
 (15, 112, 'Books', 1300, '2025-01-08')]

In [ ]:
print(orders_rdd.count())
print('=='*20)
print(*orders_rdd.take(5), sep = '\n')
print('=='*20)
type(orders_rdd)
print('=='*20)


In [9]:
orders_df = orders_rdd.toDF(
    [
        "order_id", "user_id", "category", "amount", "order_date"
])

orders_df.show()

+--------+-------+-----------+------+----------+
|order_id|user_id|   category|amount|order_date|
+--------+-------+-----------+------+----------+
|       1|    101|Electronics|  1200|2025-01-01|
|       2|    102|   Clothing|   700|2025-01-01|
|       3|    103|Electronics|   300|2025-01-02|
|       4|    101|       Home|  1500|2025-01-02|
|       5|    104|   Clothing|   400|2025-01-03|
|       6|    105|Electronics|  2200|2025-01-03|
|       7|    102|       Home|   800|2025-01-04|
|       8|    106|      Books|   250|2025-01-04|
|       9|    107|      Books|   600|2025-01-05|
|      10|    108|   Clothing|  1100|2025-01-05|
|      11|    101|Electronics|   900|2025-01-06|
|      12|    109|       Home|   300|2025-01-06|
|      13|    110|   Clothing|   950|2025-01-07|
|      14|    111|Electronics|   500|2025-01-07|
|      15|    112|      Books|  1300|2025-01-08|
+--------+-------+-----------+------+----------+



In [10]:
orders_df.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- user_id: long (nullable = true)
 |-- category: string (nullable = true)
 |-- amount: long (nullable = true)
 |-- order_date: string (nullable = true)

